# Substitution - Inference & Comparison

It loads BTE (frequency-independent) and FATE (frequency-dependent) models, performs **raw** and **consistent** inference using a sliding window, compares the results with classical solvers (Hill Climbing, Simulated Annealing), and generates plots.

**Usage:** Run the cells in order. Reduce `NUM_SAMPLES` for a quick test.

In [1]:
# === SETUP (Colab or local) ===
import sys
from pathlib import Path

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = Path('/content/drive/MyDrive/cryptanalysis-ai')
    %pip install -q pytorch-lightning torchmetrics matplotlib
else:
    REPO_DIR = Path.cwd()
    if REPO_DIR.name == 'notebooks':
        REPO_DIR = REPO_DIR.parent

SRC = REPO_DIR / 'src'
assert SRC.exists(), f'src not found: {SRC}'
sys.path.insert(0, str(SRC))
print('Repo :', REPO_DIR)


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 68.3 MB/s eta 0:00:00
Repo : /content/drive/MyDrive/cryptanalysis-ai


In [2]:
# === Load models (BTE + FATE) ===
import torch
import config
from models.substitution_bte.lightning  import SubstitutionCipherSolverBase   # BTE
from models.substitution_fate.lightning import SubstitutionCipherSolverFreq    # FATE

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(config.describe(), "\nDEVICE:", DEVICE)

# Checkpoints are auto-selected from checkpoints/<model>/.
# Or set manually: CKPT_BTE = config.CHECKPOINT_DIR/"substitution_bte"/"best-....ckpt"
CKPT_BTE  = config.find_checkpoint("substitution_bte",  prefer="best")
CKPT_FATE = config.find_checkpoint("substitution_fate", prefer="last")
assert CKPT_BTE  is not None, "BTE checkpoint not found"
assert CKPT_FATE is not None, "FATE checkpoint not found"
print("BTE :", CKPT_BTE.name)
print("FATE:", CKPT_FATE.name)

model_bte = SubstitutionCipherSolverBase.load_from_checkpoint(str(CKPT_BTE),  map_location=DEVICE)
model_fate = SubstitutionCipherSolverFreq.load_from_checkpoint(str(CKPT_FATE), map_location=DEVICE)
for m in (model_bte, model_fate):
    m.eval(); m.freeze()
print("Models are loaded.")


IN_COLAB     = True
PROJECT_ROOT = /content/drive/MyDrive/cryptanalysis-ai
STORAGE_ROOT = /content/drive/MyDrive/cryptanalysis-ai
DATA_DIR     = /content/drive/MyDrive/cryptanalysis-ai/data
CHECKPOINT   = /content/drive/MyDrive/cryptanalysis-ai/checkpoints
TB_LOG_DIR   = /content/drive/MyDrive/cryptanalysis-ai/tb_logs 
DEVICE: cuda
BTE : bte.ckpt
FATE: fate.ckpt
Models are loaded.


In [3]:
# === Validation text ===
VALIDATION_FILE = config.data_file("validation_set.txt")
with open(VALIDATION_FILE, "r", encoding="utf-8") as f:
    validation_text = f.read().replace("\n", " ")
print(f"Validation text: {len(validation_text):,} character")

TEST_LENGTHS = [30, 50, 75, 100, 200, 256, 300, 400, 500, 1000]
NUM_SAMPLES  = 500   # lower for a quick run (e.g. 50)


Validation text: 4,796,873 character


In [4]:
# === Model comparison (raw + consistent) ===
from models.substitution_common import evaluation as ev

models = {"BTE": model_bte, "FATE": model_fate}

results = ev.run_model_comparison(
    models, validation_text, TEST_LENGTHS,
    alphabet_chars=config.ALPHABET,
    num_samples=NUM_SAMPLES, seq_len=256, stride=200, device=DEVICE,
)
ev.print_comparison_table(results)


UZUNLUK  | BTE_R%  BTE_R_ms BTE_C%  BTE_C_ms | FATE_R% FATE_R_ms FATE_C% FATE_C_ms
----------------------------------------------------------------------------------
30       | 61.09   3.94     60.05   2.82     | 62.12   3.64     61.01   3.23    
50       | 78.03   2.76     77.36   2.78     | 78.73   3.17     77.99   3.20    
75       | 89.61   2.79     89.70   2.82     | 89.93   3.20     89.91   3.24    
100      | 94.40   2.79     95.25   2.83     | 94.64   3.20     95.46   3.24    
200      | 97.92   2.85     99.38   2.90     | 98.02   3.25     99.52   3.32    
256      | 97.75   2.82     99.78   2.88     | 97.79   3.23     99.83   3.29    
300      | 99.20   5.54     99.78   5.66     | 99.29   6.38     99.81   6.50    
400      | 99.67   5.54     99.87   5.67     | 99.72   6.37     99.82   6.49    
500      | 99.42   8.32     99.92   8.48     | 99.48   9.59     99.92   9.72    
1000     | 99.58   13.80    99.97   14.04    | 99.63   15.92    99.98   16.11   


In [5]:
import numpy as np

# Using mean consistent accuracies for each TEST_LENGTH
acc_fate = np.array(results["FATE"]["cons"])
acc_bte = np.array(results["BTE"]["cons"])

d = acc_fate - acc_bte
rng = np.random.default_rng(0)
n = len(d) # This will be 10, not 500, due to available data
boot = np.array([d[rng.integers(0, n, n)].mean() for _ in range(10000)])
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"mean diff {d.mean()*100:+.2f} pts, 95% CI [{lo*100:+.2f}, {hi*100:+.2f}]")

mean diff +21.83 pts, 95% CI [+5.39, +43.27]


In [6]:
import inspect
from models.substitution_common import evaluation as ev

# Let's inspect the source code of the comparison function to see
# how it generates data and calculates accuracy per sample.
print(inspect.getsource(ev.run_model_comparison))

def run_model_comparison(models, validation_text, test_lengths,
                         alphabet_chars, num_samples=500, seq_len=256,
                         stride=200, device="cpu"):
    """Compare N models.

    models : {"FATE": model_fate, "BTE": model_bte}  (Lightning module or nn.Module)
    Returns: results[name] = {"len": [...], "raw": [...], "cons": [...],
                              "raw_ms": [...], "cons_ms": [...]}
    """
    char2idx, idx2char = infer.build_vocab(alphabet_chars)
    results = {name: {"len": [], "raw": [], "cons": [], "raw_ms": [], "cons_ms": []}
               for name in models}

    for chunk_len in test_lengths:
        agg = {name: dict(raw_c=0, cons_c=0, raw_ms=0.0, cons_ms=0.0) for name in models}
        total_chars = 0

        for i in range(num_samples):
            key_map = get_fixed_key(i, alphabet_chars)
            table = str.maketrans(key_map)
            plain = _slice(validation_text, i, chunk_len)
            cipher = plain.transl

In [7]:
import numpy as np
from models.substitution_common import evaluation as ev
import config

char2idx, idx2char = ev.infer.build_vocab(config.ALPHABET)
rng = np.random.default_rng(0)
num_samples = NUM_SAMPLES

print(f"computing per-sample scores and the bootstrap of (FATE - BTE) over {num_samples} samples...")
print("-" * 125)
print(f"{'LEN':<5} | {'F_RAW %':<8} {'B_RAW %':<8} {'FARK_RAW':<12} {'95% CI_RAW':<16} | {'F_CONS %':<8} {'B_CONS %':<8} {'FARK_CONS':<12} {'95% CI_CONS'}")
print("-" * 125)

for chunk_len in TEST_LENGTHS:
    raw_fate_list, cons_fate_list = [], []
    raw_bte_list, cons_bte_list = [], []

    for i in range(num_samples):
        # generate the samples
        key_map = ev.get_fixed_key(i, config.ALPHABET)
        table = str.maketrans(key_map)
        plain = ev._slice(validation_text, i, chunk_len)
        cipher = plain.translate(table)
        m = len(plain)

        # BTE (Raw & Consistent)
        raw_bte = ev.infer.infer_raw(model_bte, cipher, char2idx, idx2char, 256, 200, DEVICE)
        cons_bte = ev.infer.infer_consistent(model_bte, cipher, char2idx, idx2char, 256, 200, DEVICE)
        raw_bte_list.append(sum(a == b for a, b in zip(plain, raw_bte[:m])) / m)
        cons_bte_list.append(sum(a == b for a, b in zip(plain, cons_bte[:m])) / m)

        # FATE (Raw & Consistent)
        raw_fate = ev.infer.infer_raw(model_fate, cipher, char2idx, idx2char, 256, 200, DEVICE)
        cons_fate = ev.infer.infer_consistent(model_fate, cipher, char2idx, idx2char, 256, 200, DEVICE)
        raw_fate_list.append(sum(a == b for a, b in zip(plain, raw_fate[:m])) / m)
        cons_fate_list.append(sum(a == b for a, b in zip(plain, cons_fate[:m])) / m)

    # convert to arrays
    raw_fate, cons_fate = np.array(raw_fate_list), np.array(cons_fate_list)
    raw_bte, cons_bte = np.array(raw_bte_list), np.array(cons_bte_list)

    # Bootstrap Analizi (RAW)
    d_raw = raw_fate - raw_bte
    n = len(d_raw)
    boot_raw = np.array([d_raw[rng.integers(0, n, n)].mean() for _ in range(10000)])
    lo_raw, hi_raw = np.percentile(boot_raw, [2.5, 97.5])

    # Bootstrap Analizi (CONS)
    d_cons = cons_fate - cons_bte
    boot_cons = np.array([d_cons[rng.integers(0, n, n)].mean() for _ in range(10000)])
    lo_cons, hi_cons = np.percentile(boot_cons, [2.5, 97.5])

    # report
    f_raw_m, b_raw_m, d_raw_m = raw_fate.mean()*100, raw_bte.mean()*100, d_raw.mean()*100
    f_cons_m, b_cons_m, d_cons_m = cons_fate.mean()*100, cons_bte.mean()*100, d_cons.mean()*100

    print(f"{chunk_len:<5} | {f_raw_m:<8.2f} {b_raw_m:<8.2f} {d_raw_m:<+12.2f} [{lo_raw*100:+.2f}, {hi_raw*100:+.2f}] | "
          f"{f_cons_m:<8.2f} {b_cons_m:<8.2f} {d_cons_m:<+12.2f} [{lo_cons*100:+.2f}, {hi_cons*100:+.2f}]")

print("-" * 125)


500 örnek için tekil skorlar ve Bootstrap (FATE - BTE) hesaplanıyor...
-----------------------------------------------------------------------------------------------------------------------------
LEN   | F_RAW %  B_RAW %  FARK_RAW     95% CI_RAW       | F_CONS % B_CONS % FARK_CONS    95% CI_CONS
-----------------------------------------------------------------------------------------------------------------------------
30    | 62.12    61.09    +1.03        [+0.03, +2.03] | 61.01    60.05    +0.96        [-0.12, +2.08]
50    | 78.73    78.03    +0.70        [-0.15, +1.56] | 77.99    77.36    +0.62        [-0.38, +1.62]
75    | 89.93    89.61    +0.32        [-0.21, +0.83] | 89.91    89.70    +0.21        [-0.41, +0.81]
100   | 94.64    94.40    +0.24        [-0.04, +0.53] | 95.46    95.25    +0.21        [-0.18, +0.61]
200   | 98.02    97.92    +0.10        [+0.02, +0.19] | 99.52    99.38    +0.14        [+0.02, +0.29]
256   | 97.79    97.75    +0.03        [-0.01, +0.07] | 99.83    9

In [ ]:
# === Classical solver baselines ===
# Higher accuracy = longer runtime. Lower NUM_SAMPLES for a quick run.
hc = ev.run_classical_eval("hill_climbing", validation_text, TEST_LENGTHS,
                           alphabet_chars=config.ALPHABET, num_samples=NUM_SAMPLES,
                           max_iterations=8000, restarts=3)
sa = ev.run_classical_eval("simulated_annealing", validation_text, TEST_LENGTHS,
                           alphabet_chars=config.ALPHABET, num_samples=NUM_SAMPLES,
                           max_iter=15000, restarts=2)
print(f"{'LEN':<8} | {'HC %':<8} {'HC ms':<10} | {'SA %':<8} {'SA ms':<10}")
print("-" * 52)
for L in TEST_LENGTHS:
    print(f"{L:<8} | {hc[L]['acc']:<8.2f} {hc[L]['avg_ms']:<10.2f} | "
          f"{sa[L]['acc']:<8.2f} {sa[L]['avg_ms']:<10.2f}")


In [ ]:
# === Plot ===
import matplotlib.pyplot as plt
import numpy as np

lengths = results["FATE"]["len"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Substitution Cipher - Model Comparison", fontsize=13, fontweight="bold")

ax = axes[0]
ax.plot(lengths, results["FATE"]["raw"],  "o--", color="#93B4E8", lw=1.5, ms=5, label="FATE - Raw")
ax.plot(lengths, results["FATE"]["cons"], "o-",  color="#2563EB", lw=2,   ms=6, label="FATE - Consistent")
ax.plot(lengths, results["BTE"]["raw"],   "s--", color="#FBB96E", lw=1.5, ms=5, label="BTE - Raw")
ax.plot(lengths, results["BTE"]["cons"],  "s-",  color="#D97706", lw=2,   ms=6, label="BTE - Consistent")
ax.set_xscale("log"); ax.set_xticks(lengths); ax.set_xticklabels(lengths, fontsize=8)
ax.set_xlabel("Text length (character)"); ax.set_ylabel("Character accuracy (%)")
ax.set_title("Accuracy by Text Length"); ax.legend(fontsize=9)
ax.grid(True, alpha=0.3); ax.axvline(x=256, color="gray", linestyle=":", lw=1, alpha=0.6)

ax2 = axes[1]
dF = [c - r for r, c in zip(results["FATE"]["raw"], results["FATE"]["cons"])]
dB = [c - r for r, c in zip(results["BTE"]["raw"],  results["BTE"]["cons"])]
x = np.arange(len(lengths)); w = 0.35
ax2.bar(x - w/2, dF, w, color="#2563EB", alpha=0.85, label="FATE")
ax2.bar(x + w/2, dB, w, color="#D97706", alpha=0.85, label="BTE")
ax2.set_xticks(x); ax2.set_xticklabels(lengths, fontsize=8)
ax2.set_xlabel("Text length (character)"); ax2.set_ylabel("Delta Accuracy (Consistent - Raw) %")
ax2.set_title("Majority-Vote Post-Processing Contribution"); ax2.legend(fontsize=9)
ax2.axhline(y=0, color="black", lw=0.8); ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
out = config.STORAGE_ROOT / "substitution_graph.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
print("Graph is saved:", out)
plt.show()
